# OrbitWars — BC v13.3bc Training

**Architecture change vs v13.1/v13.2:** Decomposed action head.
- `hold_head` → sigmoid → hold fraction per planet (MSE loss, all owned planet-slots)
- `dest_q/dest_k` → dot-product logits over 60 planets + 4 DS bins → softmax (CE loss, action planet-slots only)
- Temperature head removed; DS sincos angle logic unchanged
- Output layout per planet: `[hold(1) | dest_logits(64) | ds_sincos(8)]` = 73

**Why:** The v13.1 model collapsed to always-hold because 91.5% of planet-slots have hold targets,
drowning the launch signal. Separating hold (sigmoid) from destination (softmax) means the
destination CE loss only trains on action planet-slots — zero imbalance by design.
Empirical precedent: v5 sigmoid gate was the only prior version that learned to hold fire.

**Transfer:** v13 SA blocks kept (154 leaves); new head + CA MLPs reinitialized (30 leaves).

**Before running:** upload:
- `.env` → `/content/.env`
- `scripts/bc_train.py` → `/content/scripts/bc_train.py`
- `core/networks.py` → `/content/core/networks.py`
- rest of `core/` → `/content/core/`

In [1]:
import os, sys
from dotenv import load_dotenv
load_dotenv()

WORK_DIR  = '/content'
REPO_DIR  = '/content'
BC_DATA   = f'{WORK_DIR}/bc_data_v13_1.npz'
HIST_PATH = f'{WORK_DIR}/weights_bc_v13_3_history.json'

assert os.path.exists(f'{REPO_DIR}/scripts/bc_train.py'), 'Upload scripts/bc_train.py first'
assert os.path.exists(f'{REPO_DIR}/core/networks.py'),    'Upload core/ first'
assert os.path.exists(f'{REPO_DIR}/.env'),                'Upload .env first'

sys.path.insert(0, REPO_DIR)
print('Paths OK')

Paths OK


In [2]:
!pip uninstall -y jaxlib jax-cuda12-plugin jax-cuda12-pjrt
!pip install -q -U jax[cuda13] flax optax polars python-dotenv awscli tqdm kaggle
import jax
print('JAX devices:', jax.devices())

Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: jax-cuda12-plugin 0.7.2
Uninstalling jax-cuda12-plugin-0.7.2:
  Successfully uninstalled jax-cuda12-plugin-0.7.2
Found existing installation: jax-cuda12-pjrt 0.7.2
Uninstalling jax-cuda12-pjrt-0.7.2:
  Successfully uninstalled jax-cuda12-pjrt-0.7.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.1/525.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.4/833.4 kB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 21.8 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 133.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 260.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.5/128.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# ── Download bc_data v13.1 from Kaggle (same data as v13.1/v13.2) ─────────────
if not os.path.exists(BC_DATA):
    !KAGGLE_API_TOKEN={os.environ['KAGGLE_API_TOKEN']} kaggle datasets download -d raameshb/orbit-wars-bc-data --file bc_data_v13_1.npz -p {WORK_DIR} --unzip
    print(f'Dataset ready: {os.path.getsize(BC_DATA)/1e6:.1f} MB')
else:
    print(f'Already present: {BC_DATA} ({os.path.getsize(BC_DATA)/1e6:.1f} MB)')

Dataset URL: https://www.kaggle.com/datasets/raameshb/orbit-wars-bc-data
License(s): CC0-1.0
100% 377M/377M [00:10<00:00, 36.6MB/s] 

Dataset ready: 395.1 MB


In [4]:
# ── Download v13.3bc resume checkpoint from R2 ────────────────────────────────
RESUME_PATH = f'{WORK_DIR}/weights_bc_v13_3_resume.pkl'
if not os.path.exists(RESUME_PATH):
    !AWS_ACCESS_KEY_ID={os.environ['R2_ACCESS_KEY_ID']} AWS_SECRET_ACCESS_KEY={os.environ['R2_SECRET_ACCESS_KEY']} AWS_DEFAULT_REGION=auto aws s3 cp s3://{os.environ['R2_BUCKET_NAME']}/bc_v13_3bc/weights_bc_v13_3_resume.pkl {RESUME_PATH} --endpoint-url {os.environ['R2_ENDPOINT_URL']}
    print(f'Downloaded → {RESUME_PATH} ({os.path.getsize(RESUME_PATH)/1e3:.1f} KB)')
else:
    print(f'Already present: {RESUME_PATH}')

download: s3://orbit-wars-checkpoints/bc_v13_3bc/weights_bc_v13_3_resume.pkl to ./weights_bc_v13_3_resume.pkl
Downloaded → /content/weights_bc_v13_3_resume.pkl (1711.1 KB)


In [ ]:
ACTOR_OUT = f'{WORK_DIR}/weights_bc_v13_3.pkl'

# Warm restart from epoch-150 checkpoint: batch 4096, lr 4e-2, 150 more epochs
!python {REPO_DIR}/scripts/bc_train.py --train --bc-data {BC_DATA} --out {ACTOR_OUT} --epochs 150 --batch-size 4096 --lr 4e-2 --warmup-epochs 5 --weight-decay 0 --hold-rate 1.0 --ffa-weight 3 --eval-every 5 --checkpoint-every 30 --r2-prefix bc_v13_3bc --resume

In [ ]:
# ── Loss curves ───────────────────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

with open(HIST_PATH) as f:
    hist = json.load(f)

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('OrbitWars BC v13.3bc — Decomposed Action Head', fontsize=14, fontweight='bold')

if 'actor' in hist:
    epochs     = [r['epoch']      for r in hist['actor']]
    train_loss = [r['train_loss'] for r in hist['actor']]
    val_loss   = [r['val_loss']   for r in hist['actor']]

    ax.plot(epochs, train_loss, label='train', linewidth=1.5, alpha=0.9)
    ax.plot(epochs, val_loss,   label='val',   linewidth=1.5, alpha=0.9, linestyle='--')

    best_ep  = epochs[int(np.argmin(val_loss))]
    best_val = min(val_loss)
    ax.axvline(best_ep, color='gray', linestyle=':', linewidth=1, alpha=0.6)
    ax.annotate(f'best val\n{best_val:.4f}',
                xy=(best_ep, best_val), xytext=(best_ep + len(epochs)*0.04, best_val),
                fontsize=8, color='gray',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (hold MSE + dest CE)')
ax.legend()
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = f'{WORK_DIR}/loss_curves_v13_3.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {plot_path}')

In [ ]:
# ── Verify outputs ────────────────────────────────────────────────────────────
for path in [ACTOR_OUT, plot_path]:
    if os.path.exists(path):
        print(f'{path}: {os.path.getsize(path)/1e3:.1f} KB')
    else:
        print(f'{path}: NOT FOUND')